[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/e_series/e_series.ipynb)

# The E Series

This example builds the exceptional root systems E6, E7 and E8, each a configuration of equal-length vectors preserved by reflection in the hyperplane square to any of its roots, and each nested in the next: one more generating mirror extends E6 to E7, and another E7 to E8. Seen in E8's Coxeter plane, all three appear at once, the smaller ones among the roots of the larger.

The mirrors are vectors of one eight-dimensional algebra, and their reflections are maps on vectors, `mirror >> Vector`; composed, they are each system's Coxeter motion, which carries one seed per orbit through all the roots, 72, 126 and 240. Averaged with its adjoint, `(action + action.adjoint()) / 2`, the motion gives its planes of turning by one eigensolve. The sum of the roots' dyads, `roots * (roots | Vector)`, is a multiple of the projector onto the space they span, so composing these projectors shows each system inside the next. E7 is composed of an odd number of reflections: it turns three planes and reverses one more direction, and is no rotor.

In the notation of Lie theory the Coxeter numbers of E6, E7 and E8 are $h = 12, 18, 30$ and their exponents $1, 4, 5, 7, 8, 11$, then $1, 5, 7, 9, 11, 13, 17$, then $1, 7, 11, 13, 17, 19, 23, 29$; the planes turn by $m$ and $h - m$, and E7's exponent $h / 2 = 9$ is its reversed direction.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.animation import save_figure
from examples.math.e8.core import orbit
from examples.math.e_series import render

np.set_printoptions(precision=6, suppress=True)

In [ ]:
ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
VectorMap = ga.gatype((Vector, Vector))

## 1. Generating mirrors

The simple roots specify the generating mirrors. Each node below names one root; connected normals meet at 120 degrees, and unconnected normals are perpendicular. The first six roots give E6. Extending the right arm once gives E7, and twice gives E8. Their ranks count independent root directions within the shared ambient space.

In [ ]:
names = ("E6", "E7", "E8")
ranks = np.array([6, 7, 8])
periods = np.array([12, 18, 30])

simple_roots = stack([
    (mv.a + mv.h - mv.b - mv.c - mv.d - mv.e - mv.f - mv.g) / 2,
    mv.a + mv.b,
    mv.b - mv.a,
    mv.c - mv.b,
    mv.d - mv.c,
    mv.e - mv.d,
    mv.f - mv.e,
    mv.g - mv.f,
])                                                                          # [mirrors] Vector

display(Image(filename=str(save_figure(
    render.draw_diagram(simple_roots, ranks, names), "e_series_diagram",
))))

## 2. Root orbits

Composing one reflection from every simple root gives a Coxeter motion. It repeats after 12, 18 or 30 steps for E6, E7 or E8. One carefully chosen starting root per generator gives disjoint orbits that together exhaust the root system: 72, 126 and 240 roots.

In [ ]:
def orbit_starts(simple: Vector) -> Iterator[Vector]:
    """One root from each orbit, acted on by the reflections preceding it."""
    preceding = Vector                                                       # Vector <- Vector
    for root in simple:
        yield preceding(root)                                                # Vector
        preceding = preceding(root.normalized() >> Vector)                   # Vector <- Vector


def root_system(simple: Vector, n_steps: int) -> tuple[VectorMap, Vector]:
    reflections = simple.normalized() >> Vector                              # [mirrors] Vector <- Vector
    # Right composition matches the prefix order used to choose the starting roots.
    action = reduce(lambda left, right: left(right), reflections)             # Vector <- Vector
    seeds = stack(orbit_starts(simple))                                       # [orbits] Vector
    roots = stack(orbit(action, seeds, n_steps))                               # [steps, orbits] Vector
    return action, roots


systems = tuple(root_system(simple_roots[:n_rank], n_steps)
                for n_rank, n_steps in zip(ranks, periods))
actions = stack([action for action, _ in systems])                            # [cases] Vector <- Vector
root_systems = tuple(roots for _, roots in systems)

## 3. One shared projection

All three systems appear in E8's Coxeter plane. Teal marks the E6 roots, amber those added by E7, and purple those added by E8; edges join nearest neighbours in eight dimensions. Averaging a motion with its inverse scales each rotation plane by the cosine of its angle. Paired eigendirections recover those planes, with the slowest turn defining the Coxeter plane.

In [ ]:
def rotation_planes(action: VectorMap, n_rank: int) -> tuple[Vector, Vector, Scalar]:
    symmetric = (action + action.adjoint()) / 2                               # Vector <- Vector
    cosines, directions = symmetric.eigh()                                    # [directions] Scalar, Vector

    # Unused ambient directions have cosine one and sort last.
    # Paired cosines describe turns; a lone minus one describes a reflection.
    plane_cosines = cosines[n_rank - 1:0:-2]                                  # [planes] Scalar
    horizontal = directions[n_rank - 1:0:-2]                                  # [planes] Vector
    vertical = (action(horizontal) - plane_cosines * horizontal).normalized() # [planes] Vector
    return horizontal, vertical, plane_cosines.arccos()


plane_frames = tuple(rotation_planes(action, n_rank)
               for (action, _), n_rank in zip(systems, ranks))
horizontal, vertical, _ = plane_frames[-1]                                  # [planes] Vector, Vector, Scalar

display(Image(filename=str(save_figure(
    render.draw_nested_roots(root_systems, horizontal[0], vertical[0], names),
    "e_series_nested",
))))

## 4. The spaces inside the pictures

Each root contributes a dyad that measures along that root and responds along it. Their sum has the same strength in every direction of the root span. Dividing by that strength gives the orthogonal projector onto the span. Composing adjacent projectors retains the smaller space: E6 sits inside E7, which sits inside E8.

In [ ]:
# Root lengths squared are two; each system has rank times period roots.
spans = stack([
    (roots * (roots | Vector)).sum(axis=(0, 1)) / (2 * n_steps)
    for roots, n_steps in zip(root_systems, periods)
])                                                                          # [cases] Vector <- Vector
shared_spans = spans[:-1](spans[1:])                                         # [adjacent cases] Vector <- Vector

print("Root-span dimensions, E6 / E7 / E8:")
print(render.format_scalars(spans.trace()))
print("Shared dimensions, E6 with E7 / E7 with E8:")
print(render.format_scalars(shared_spans.trace()))

## 5. The reflection in E7

The E6 and E8 motions consist entirely of turns in perpendicular planes. E7 has three such planes and one remaining direction that changes sign at every step. Its seven generating reflections therefore cannot combine into a rotor. The planar picture shows only one part of this motion.

In [ ]:
# The plane dyads account for every turning direction.
turning_spans = stack([
    (plane_x * (plane_x | Vector) + plane_y * (plane_y | Vector)).sum(axis=0)
    for plane_x, plane_y, _ in plane_frames
])                                                                          # [cases] Vector <- Vector
reversing_spans = spans - turning_spans                                       # [cases] Vector <- Vector

print("Reversed dimensions, E6 / E7 / E8:")
print(render.format_scalars(reversing_spans.trace()))
for name, (_, _, angles), n_steps in zip(names, plane_frames, periods):
    winding_rates = angles * n_steps / (2 * np.pi)                             # [planes] Scalar
    print(f"{name}: turns in each plane over one complete orbit")
    print(render.format_scalars(winding_rates))

These reflections generate the finite Weyl groups associated with E6, E7 and E8. The root systems also organize the continuous Lie algebras bearing those names; their Lie-algebra dimensions are 78, 133 and 248. John Stembridge’s [Coxeter-plane account](https://websites.umich.edu/~jrs/coxplane.html) explains the circular projections, and the [MIT classification notes](https://math.mit.edu/~dav/roots.pdf) place these diagrams among the other root families.